# should_speak

PAL gets a stream of events: **speech** (transcribed) and **scene** (camera descriptions).
For each event the code picks one question, so every decision is a single model call:

| Event | PAL speaks when | Choices |
|---|---|---|
| speech | the user addresses PAL, or PAL can clearly help | `addressed` / `can_help` / `not_for_pal` |
| scene | there is danger or a watch request fires, or PAL can clearly help | `warn` / `can_help` / `nothing` |

The state stays small: standing watch requests, a few retrieved memories, the last few exchanges, and the event itself (last, so a stable prefix can be cached).
Keep it well under Ollama's default 2048-token context for tev1; latency grows about 3 ms per token on the 4b.

In [ ]:
import time
import uuid
from dataclasses import dataclass

from pal.llm import get_client

client = get_client('typesafe:tev1:4b')

## State

In [ ]:
def make_state(event, scene=None, watching_for=(), memories=(), recent=()):
    """Stable context first, the new event last."""
    state = {'watching_for': list(watching_for), 'memories': list(memories), 'recent': list(recent)}
    if scene is not None:
        state['scene'] = scene
    state['event'] = event
    return state


def heard(text, scene, speaker='user', **context):
    return make_state({'type': 'speech', 'speaker': speaker, 'text': text}, scene=scene, **context)


def seen(description, **context):
    return make_state({'type': 'scene', 'description': description}, **context)

## Questions

In [ ]:
STATE_DESCRIPTION = (
    "The state describes a moment for a user wearing PAL, a voice assistant with a camera and "
    "microphone. 'watching_for' lists things the user asked PAL to watch for, 'memories' are "
    "things PAL observed earlier that may be relevant, 'recent' is the conversation so far, "
    "and 'event' is what just happened. "
)

SPEECH_QUESTIONS = {
    'should_speak': {
        'type': 'choice',
        'instructions': STATE_DESCRIPTION + (
            "The event is speech that was just heard; 'speaker' is who said it and 'scene' is what "
            "the camera currently sees. Should PAL speak in response to this speech?"
        ),
        'criteria': {
            'addressed': (
                "The user is speaking to PAL: they call PAL by name to ask or tell it something, "
                "continue a conversation PAL is part of, or give an assistant-style request while "
                "nobody else is around."
            ),
            'can_help': (
                "The speech is not meant for PAL, but the user is stuck on something PAL can clearly "
                "resolve from its memories or the scene, such as looking for an object PAL has seen."
            ),
            'not_for_pal': (
                "The speech is meant for another person, is someone other than the user talking, is "
                "small talk, merely mentions PAL, or is the user thinking aloud about something PAL "
                "cannot help with."
            ),
        },
    },
}

SCENE_QUESTIONS = {
    'should_speak': {
        'type': 'choice',
        'instructions': STATE_DESCRIPTION + (
            "The event is a new description of what the camera sees. Should PAL say something to "
            "the user about it, without being asked?"
        ),
        'criteria': {
            'warn': (
                "The scene shows a real danger to the user or others, or something listed in "
                "'watching_for' is happening."
            ),
            'can_help': (
                "There is no danger, but the user is visibly stuck on something PAL can clearly "
                "resolve from its memories, such as a door they cannot open while PAL knows where "
                "their keys are."
            ),
            'nothing': (
                "Nothing in the scene needs the user's attention; speaking up would only interrupt."
            ),
        },
    },
}

QUESTIONS = {'speech': SPEECH_QUESTIONS, 'scene': SCENE_QUESTIONS}

## Decision

Speak only when the top choice is a speaking reason and its probability clears that reason's bar.
`can_help` gets a higher bar: a wrong unprompted remark costs more than a missed one, since the user can still ask.

In [ ]:
# Choices that make PAL speak, with the minimum probability each needs.
MIN_PROB = {'addressed': 0.5, 'warn': 0.5, 'can_help': 0.6}


@dataclass
class Decision:
    speak: bool
    reason: str
    probability: float
    probabilities: dict[str, float]


def should_speak(state, min_prob=MIN_PROB) -> Decision:
    answer = client.decide(state, QUESTIONS[state['event']['type']]).answers['should_speak']
    p = answer.probabilities[answer.choice]
    speak = answer.choice in min_prob and p >= min_prob[answer.choice]
    return Decision(speak, answer.choice, p, answer.probabilities)

## Test cases

In [ ]:
KEYS = '10:42 saw keys on the hallway side table'
PARKED = '07:58 user parked on level 2, spot B14'
SERVER_ROOM = 'anyone entering the server room'

# (expected choice, state)
SPEECH_CASES = [
    # addressed
    ('addressed',   heard('hey PAL, what time is it?', 'office desk, user alone')),
    ('addressed',   heard('and the one after that?', 'office desk, user alone',
                          recent=['user: PAL, when is my next meeting?', 'PAL: Standup at 10:00.'])),
    ('addressed',   heard('PAL, log that the back gate was unlocked', 'back gate, user on patrol')),
    ('addressed',   heard('remind me to call my mom at five', 'kitchen, user alone')),
    ('addressed',   heard("hold on... PAL, what's my schedule on Thursday?", 'user on a phone call')),
    ('addressed',   heard('PAL, stop. stop talking.', 'office desk, user alone',
                          recent=['PAL: Here are the five most recent incidents. The first one'])),
    # can_help: not meant for PAL, but PAL knows the answer
    ('can_help',    heard('ugh, where did I leave my keys', 'hallway, user patting their pockets', memories=[KEYS])),
    ('can_help',    heard('where did I park this morning...', 'parking garage, user looking around', memories=[PARKED])),
    ('can_help',    heard("no idea, I forgot his name too", 'guard booth, colleague next to the user', speaker='colleague',
                          recent=["user: what was the new guard's name again?"],
                          memories=['09:15 user said the new guard is called Ahmed'])),
    # not_for_pal
    ('not_for_pal', heard('ugh, where did I leave my keys', 'hallway, user patting their pockets')),  # PAL can't help
    ('not_for_pal', heard('yeah, see you tomorrow Mark!', 'office exit, a colleague waving')),
    ('not_for_pal', heard('did you see the game last night?', 'coffee machine, two people chatting', speaker='colleague')),
    ('not_for_pal', heard('okay, two sugars please', 'cafe counter, a barista')),
    ('not_for_pal', heard('hmm, left or right... left', 'street corner, user alone')),
    ('not_for_pal', heard('yeah PAL is honestly great, you should get one', 'user on a phone call')),
    ('not_for_pal', heard('PAL, open the door', 'reception, a visitor speaking toward the user', speaker='visitor')),
    ('not_for_pal', heard('breaking news: heavy rain expected tonight', 'break room, TV on', speaker='tv')),
    ('not_for_pal', heard('can you hand me that radio?', 'a colleague standing next to the user')),
    ('not_for_pal', heard('all clear on level two, over', 'user speaking into a handheld radio')),
    ('not_for_pal', heard("let's ask PAL about that later", 'a colleague standing next to the user')),
]

SCENE_CASES = [
    # warn
    ('warn',     seen('smoke is coming from under the storage room door')),
    ('warn',     seen('a forklift is reversing toward the user, who is facing away')),
    ('warn',     seen('a person in a hoodie is opening the server room door', watching_for=[SERVER_ROOM])),
    ('warn',     seen('a backpack has sat alone next to the bench for ten minutes', watching_for=['unattended bags'])),
    ('warn',     seen('user talking to a visitor; a car with nobody in it is rolling toward them')),
    ('warn',     seen('a person is lying motionless at the bottom of the stairwell')),
    # can_help
    ('can_help', seen('user at the front door, trying the handle and searching their pockets', memories=[KEYS])),
    ('can_help', seen('user walking up and down parking level 3, looking at the cars', memories=[PARKED])),
    # nothing
    ('nothing',  seen('quiet empty corridor, nothing unusual')),
    ('nothing',  seen('a colleague walks past the server room and nods at the user', watching_for=[SERVER_ROOM])),
    ('nothing',  seen('user walks past the hallway side table toward the kitchen', memories=[KEYS])),  # not stuck
    ('nothing',  seen('user unlocks the front door and walks out', memories=[KEYS])),
    ('nothing',  seen('reception, a visitor signing in while the user watches')),
    ('nothing',  seen('break room TV is showing a car chase from an action movie')),
]

## Evaluate

Marks: `ok` right choice, `~` right speak/silent but wrong reason, `MISS` stayed silent when it should speak, `LOUD` spoke when it should stay silent (the expensive one).
Each state gets a fresh `event.id`, so Ollama's cache can't make latency look better than it is.

In [ ]:
def evaluate(cases, min_prob=MIN_PROB):
    counts = {'ok': 0, '~': 0, 'MISS': 0, 'LOUD': 0}
    times = []
    for expected, state in cases:
        state = dict(state, event=dict(state['event'], id=uuid.uuid4().hex[:8]))
        start = time.perf_counter()
        d = should_speak(state, min_prob)
        times.append(time.perf_counter() - start)

        if d.speak != (expected in min_prob):
            mark = 'LOUD' if d.speak else 'MISS'
        else:
            mark = 'ok' if d.reason == expected else '~'
        counts[mark] += 1

        event = state['event']
        print(f"{mark:4} {d.reason:11} p={d.probability:.2f} {'speak ' if d.speak else 'silent'}  "
              f"expected={expected:11}  {event.get('text') or event.get('description')!r}")

    print('\n' + ', '.join(f'{k} {v}' for k, v in counts.items())
          + f' of {len(cases)}, mean latency {1000 * sum(times) / len(times):.0f} ms')

In [ ]:
evaluate(SPEECH_CASES)

In [ ]:
evaluate(SCENE_CASES)